## 1. Setup and Data Generation

First, let's import the necessary libraries and create two types of dummy datasets: one for general classification tasks (suitable for K-Fold, Stratified K-Fold, and LOOCV) and one for time series (for Time Series Split).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut, TimeSeriesSplit, train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# --- Regression Dataset (for K-Fold, LOOCV) ---
X_reg, y_reg = make_regression(
    n_samples=200, n_features=2, n_informative=2, n_targets=1,
    noise=20, random_state=42
)

print(f"Regression Dataset Shape: X={X_reg.shape}, y={y_reg.shape}")

# --- Time Series Dataset ---
# Create a simple linear trend with some noise
n_samples_ts = 150
time = np.arange(n_samples_ts)
# Increased slope and reduced noise for a clearer trend
dependent_variable = 10 * time + np.random.normal(0, 20, n_samples_ts)

X_ts = time.reshape(-1, 1) # Feature is just time
y_ts = dependent_variable # Target is the dependent variable

print(f"\nTime Series Dataset Shape: X={X_ts.shape}, y={y_ts.shape}")

Regression Dataset Shape: X=(200, 2), y=(200,)

Time Series Dataset Shape: X=(150, 1), y=(150,)


## 2. Basic Train-Test Split (General Cross-Validation Concept)

Before diving into specific cross-validation techniques, let's briefly recall the basic concept: splitting data into training and testing sets to evaluate model generalization. While not a 'cross-validation' method in itself, it's the fundamental idea from which CV methods evolve.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

model_basic = LinearRegression()
model_basic.fit(X_train, y_train)
y_pred_basic = model_basic.predict(X_test)

from sklearn.metrics import r2_score
r2_basic = r2_score(y_test, y_pred_basic)
print(f"Basic Train-Test Split R2 Score: {r2_basic:.4f}")

Basic Train-Test Split R2 Score: 0.8595


## 3. K-Fold Cross-Validation

K-Fold Cross-Validation divides the dataset into `k` folds. The model is trained `k` times, each time using one fold as the test set and the remaining `k-1` folds as the training set. The results are then averaged.

In [ ]:
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

r2_scores_kf = []
fold_idx = 1

print(f"Performing {k_folds}-Fold Cross-Validation...")
for train_index, test_index in kf.split(X_reg):
    X_train_kf, X_test_kf = X_reg[train_index], X_reg[test_index]
    y_train_kf, y_test_kf = y_reg[train_index], y_reg[test_index]

    model_kf = LinearRegression()
    model_kf.fit(X_train_kf, y_train_kf)
    y_pred_kf = model_kf.predict(X_test_kf)

    from sklearn.metrics import r2_score
    r2_current_fold = r2_score(y_test_kf, y_pred_kf)
    r2_scores_kf.append(r2_current_fold)

    print(f"  Fold {fold_idx}: R2 Score = {r2_current_fold:.4f}")
    fold_idx += 1

print(f"\nAverage K-Fold R2 Score: {np.mean(r2_scores_kf):.4f} (+/- {np.std(r2_scores_kf):.4f})")

Performing 5-Fold Cross-Validation...
  Fold 1: R2 Score = 0.8819
  Fold 2: R2 Score = 0.7865
  Fold 3: R2 Score = 0.8353
  Fold 4: R2 Score = 0.8861
  Fold 5: R2 Score = 0.7262

Average K-Fold R2 Score: 0.8232 (+/- 0.0605)


## 4. Stratified K-Fold Cross-Validation

Stratified K-Fold is a variation of K-Fold that returns stratified folds: each set of folds contains approximately the same percentage of samples of each target class as the complete set. This is particularly useful for classification tasks with imbalanced datasets.

First, let's create a classification dataset.

In [ ]:
# --- Classification Dataset (for Stratified K-Fold) ---
X_clf, y_clf = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_classes=2, weights=[0.9, 0.1], flip_y=0, random_state=42
)

print(f"Classification Dataset Shape: X={X_clf.shape}, y={y_clf.shape}")
print(f"Class distribution: {np.bincount(y_clf)}")

Classification Dataset Shape: X=(200, 2), y=(200,)
Class distribution: [180  20]


In [ ]:
k_folds_stratified = 5
skf = StratifiedKFold(n_splits=k_folds_stratified, shuffle=True, random_state=42)

accuracies_skf = []
fold_idx = 1

print(f"Performing {k_folds_stratified}-Fold Stratified Cross-Validation...")
for train_index, test_index in skf.split(X_clf, y_clf):
    X_train_skf, X_test_skf = X_clf[train_index], X_clf[test_index]
    y_train_skf, y_test_skf = y_clf[train_index], y_clf[test_index]

    model_skf = LogisticRegression(random_state=42, solver='liblinear')
    model_skf.fit(X_train_skf, y_train_skf)
    y_pred_skf = model_skf.predict(X_test_skf)

    accuracy_current_fold = accuracy_score(y_test_skf, y_pred_skf)
    accuracies_skf.append(accuracy_current_fold)

    print(f"  Fold {fold_idx}: Accuracy = {accuracy_current_fold:.4f}, Class distribution in test set: {np.bincount(y_test_skf)}")
    fold_idx += 1

print(f"\nAverage Stratified K-Fold Accuracy: {np.mean(accuracies_skf):.4f} (+/- {np.std(accuracies_skf):.4f})")

Performing 5-Fold Stratified Cross-Validation...
  Fold 1: Accuracy = 0.8750, Class distribution in test set: [36  4]
  Fold 2: Accuracy = 1.0000, Class distribution in test set: [36  4]
  Fold 3: Accuracy = 0.9500, Class distribution in test set: [36  4]
  Fold 4: Accuracy = 0.9500, Class distribution in test set: [36  4]
  Fold 5: Accuracy = 0.8500, Class distribution in test set: [36  4]

Average Stratified K-Fold Accuracy: 0.9250 (+/- 0.0548)


## 5. Leave-One-Out Cross-Validation (LOOCV)

LOOCV is an extreme case of K-Fold where `k` equals the number of samples (`n`). Each sample is used as a test set exactly once, while the remaining `n-1` samples form the training set. It's computationally intensive for large datasets.

_Note: Stratified K-Fold was omitted as it is primarily for classification problems, not regression._

In [ ]:
loo = LeaveOneOut()

mses_loo = []

# For demonstration, we'll use a smaller subset due to computational cost
# Otherwise, it would run 200 times for our X_reg dataset.
X_small_reg, y_small_reg = make_regression(
    n_samples=20, n_features=2, n_informative=2, n_targets=1,
    noise=10, random_state=42
)

print(f"Performing Leave-One-Out Cross-Validation on a subset of {X_small_reg.shape[0]} samples...")

for train_index, test_index in loo.split(X_small_reg):
    X_train_loo, X_test_loo = X_small_reg[train_index], X_small_reg[test_index]
    y_train_loo, y_test_loo = y_small_reg[train_index], y_small_reg[test_index]

    model_loo = LinearRegression()
    model_loo.fit(X_train_loo, y_train_loo)
    y_pred_loo = model_loo.predict(X_test_loo)

    # Using MSE instead of R2 for LOOCV as R2 is undefined for single-sample test sets
    mse_current_fold = mean_squared_error(y_test_loo, y_pred_loo)
    mses_loo.append(mse_current_fold)

print(f"\nAverage LOOCV MSE: {np.mean(mses_loo):.4f} (+/- {np.std(mses_loo):.4f})")
print(f"Note: LOOCV is very slow for large datasets as it performs N training runs.")

Performing Leave-One-Out Cross-Validation on a subset of 20 samples...

Average LOOCV MSE: 74.6279 (+/- 98.0756)
Note: LOOCV is very slow for large datasets as it performs N training runs.


## 6. Time Series Split

Time Series Split is specifically designed for time series data. It ensures that the training sets are always chronologically before the test sets, respecting the temporal order of the data. It creates multiple train-test pairs, where each test set is a future segment.

In [ ]:
n_splits_ts = 5
tscv = TimeSeriesSplit(n_splits=n_splits_ts)

r2_scores_ts = [] # Changed to R2 scores
fold_idx = 1

print(f"Performing Time Series Split with {n_splits_ts} splits...")
for train_index, test_index in tscv.split(X_ts):
    X_train_ts, X_test_ts = X_ts[train_index], X_ts[test_index]
    y_train_ts, y_test_ts = y_ts[train_index], y_ts[test_index]

    model_ts = LinearRegression()
    model_ts.fit(X_train_ts, y_train_ts)
    y_pred_ts = model_ts.predict(X_test_ts)

    from sklearn.metrics import r2_score # Import r2_score
    r2_current_fold = r2_score(y_test_ts, y_pred_ts) # Calculate R2 score
    r2_scores_ts.append(r2_current_fold)

    print(f"  Fold {fold_idx}: Train = {len(train_index)} samples (Time {X_ts[train_index[0]][0]:.0f} to {X_ts[train_index[-1]][0]:.0f}), Test = {len(test_index)} samples (Time {X_ts[test_index[0]][0]:.0f} to {X_ts[test_index[-1]][0]:.0f}), R2 Score = {r2_current_fold:.2f}") # Print R2 score
    fold_idx += 1

print(f"\nAverage Time Series Split R2 Score: {np.mean(r2_scores_ts):.2f} (+/- {np.std(r2_scores_ts):.2f})") # Print average R2 score

Performing Time Series Split with 5 splits...
  Fold 1: Train = 25 samples (Time 0 to 24), Test = 25 samples (Time 25 to 49), R2 Score = 0.82
  Fold 2: Train = 50 samples (Time 0 to 49), Test = 25 samples (Time 50 to 74), R2 Score = 0.89
  Fold 3: Train = 75 samples (Time 0 to 74), Test = 25 samples (Time 75 to 99), R2 Score = 0.96
  Fold 4: Train = 100 samples (Time 0 to 99), Test = 25 samples (Time 100 to 124), R2 Score = 0.93
  Fold 5: Train = 125 samples (Time 0 to 124), Test = 25 samples (Time 125 to 149), R2 Score = 0.93

Average Time Series Split R2 Score: 0.91 (+/- 0.05)


## 7. Cross-Validation Score (Simplified Workflow)

In [ ]:
from sklearn.model_selection import cross_val_score

# `cross_val_score` is a convenient function in scikit-learn that automates the process of splitting data,
# training a model, and evaluating it across multiple folds.
# It returns an array of scores, one for each fold.
#
# We'll use it with our regression dataset (`X_reg`, `y_reg`) and `LinearRegression`.

print("\nPerforming Cross-Validation using cross_val_score...")

# Define the model
model_cvs = LinearRegression()

# Perform 5-fold cross-validation
# The scoring parameter can be changed, e.g., 'neg_mean_squared_error', 'r2', etc.
# For regression, 'r2' is a common choice.
scores_cvs = cross_val_score(model_cvs, X_reg, y_reg, cv=5, scoring='r2')

print(f"  R2 Scores for each fold: {scores_cvs}")
print(f"Average cross_val_score R2: {np.mean(scores_cvs):.4f} (+/- {np.std(scores_cvs):.4f})")


Performing Cross-Validation using cross_val_score...
  R2 Scores for each fold: [0.89063601 0.84153684 0.77819236 0.7709445  0.87973788]
Average cross_val_score R2: 0.8322 (+/- 0.0499)


In [ ]:
from sklearn.metrics import get_scorer_names

In [ ]:
get_scorer_names()

['accuracy',
 'adjusted_mutual_info_score',
 'adjusted_rand_score',
 'average_precision',
 'balanced_accuracy',
 'completeness_score',
 'd2_absolute_error_score',
 'explained_variance',
 'f1',
 'f1_macro',
 'f1_micro',
 'f1_samples',
 'f1_weighted',
 'fowlkes_mallows_score',
 'homogeneity_score',
 'jaccard',
 'jaccard_macro',
 'jaccard_micro',
 'jaccard_samples',
 'jaccard_weighted',
 'matthews_corrcoef',
 'mutual_info_score',
 'neg_brier_score',
 'neg_log_loss',
 'neg_max_error',
 'neg_mean_absolute_error',
 'neg_mean_absolute_percentage_error',
 'neg_mean_gamma_deviance',
 'neg_mean_poisson_deviance',
 'neg_mean_squared_error',
 'neg_mean_squared_log_error',
 'neg_median_absolute_error',
 'neg_negative_likelihood_ratio',
 'neg_root_mean_squared_error',
 'neg_root_mean_squared_log_error',
 'normalized_mutual_info_score',
 'positive_likelihood_ratio',
 'precision',
 'precision_macro',
 'precision_micro',
 'precision_samples',
 'precision_weighted',
 'r2',
 'rand_score',
 'recall',
 're